# Suite DIA — The Python, Matlab and Excel dialects

Each dialect models a language's expressions: its kinds, its vocabulary and the domains of its values, `render`
to its source text, an evaluator with the language's own rules, and a scope that resolves names, imports and
references as the language does. Python evaluates by Python's rules, with NumPy as one more module to import;
MATLAB is two-valued over doubles, logicals and strings, with functions from the path and packages; Excel coerces,
computes with error values, and reads cells from workbooks. What every dialect shares is suite FRM; the Basic
dialect is suites EXP and EVL.

In [ ]:
import math

import numpy as np

from mbse.Expressions.Dialects.Excel import Domains as XD, Evaluators as XV, Expressions as X
from mbse.Expressions.Dialects.Matlab import Domains as MD, Evaluators as MV, Expressions as M
from mbse.Expressions.Dialects.Python import Domains as PD, Evaluators as PV, Expressions as P
from mbse.Expressions.Framework import Domains as D, Expressions as F
from mbse.Schemas.Framework import Proxies, Schemas as S
from support import raises, text

Person = S.OfObject.Builder().properties(text("name"), text("age", int), text("email")).create()
Proxies.register("Person", Person)
ann = Proxies.Builders.Person().name("Ann").age(30).email("ann@example.com").create()
bob = Proxies.Builders.Person().name("Bob").age(70).create()  # no email

## DIA-01 · Python: expressions as `ast` has them, with their imports; rendered and parsed as source

In [ ]:
t, a, b, c, d, x, y, z = (P.name(n) for n in "tabcdxyz")
for built, source in [  # written with constructors, as source
    (P.import_("numpy", P.importfrom("numpy.ma", "getmaskarray", P.call(
        "np.logical_and", P.call("np.greater_equal", P.subscript(t, "age"), 18),
        P.call("np.logical_not", P.call("getmaskarray", P.subscript(t, "email"))))), alias="np"),
     "import numpy as np\nfrom numpy.ma import getmaskarray\n"
     "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))"),
    (P.let_("a", t, P.boolop("and", P.compare(">", P.attribute(a, "b"), 1), P.unaryop("not", P.attribute(a, "c")))),
     "(lambda a: a.b > 1 and not a.c)(t)"),
    (P.ifexp(P.compare("<", a, b), x, P.unaryop("-", P.binop("**", y, 2))), "x if a < b else -y ** 2"),
    (P.ifexp(b, P.ifexp(a, x, y), z), "(x if a else y) if b else z"),
    (P.ifexp(a, x, P.ifexp(b, y, z)), "x if a else y if b else z"),
    (P.binop("-", P.binop("-", P.binop("**", 2, P.unaryop("-", 1)), P.binop("-", a, b)), c), "2 ** -1 - (a - b) - c"),
    (P.boolop("or", P.boolop("and", a, b), P.boolop("and", c, P.unaryop("not", d))), "a and b or c and not d"),
    (P.boolop("and", P.boolop("or", a, b), P.boolop("or", c, d)), "(a or b) and (c or d)"),
    (P.compare("!=", P.unaryop("not", P.compare("==", a, b)), P.compare("<=", c, d)), "(not a == b) != (c <= d)"),
    (P.call(P.call("f", P.name("g")), P.subscript(P.attribute(x, "y"), "z"), P.unaryop("+", 1), b"\x00", True, "it's"),
     "f(g)(x.y['z'], +1, b'\\x00', True, \"it's\")"),
    (P.import_("a", P.import_("b", P.importfrom("d", "e", P.importfrom("d", "f", 1, alias="g")), alias="c")),
     "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1"),
]:
    assert P.render(built) == source, (P.render(built), source)
    assert F.same(P.parse(source), built), source  # Python reads it back (parse is Python's only)
for source in [
    "import numpy as np\nfrom numpy.ma import getmaskarray\nnp.logical_and(np.greater_equal(t['age'], 18), "
    "np.logical_not(getmaskarray(t['email'])))",
    "import numpy.ma\nnumpy.ma.masked",
    "(lambda a: a.b > 1 and not a.c)(t)",
    "x if a < b else -y ** 2",
    "(x if a else y) if b else z",
    "(-1) ** 2 + 3 * (4 - 5) // 2 % 7 / 1.5",
    "2 ** -1 - (a - b) - c",
    "a and b or c and not d",
    "(a or b) and (c or d)",
    "not (a == b) != (c <= d)",
    "f(g)(x.y['z'], +1, b'\\x00', True, 'it''s')",
]:
    parsed = P.parse(source)
    assert P.render(parsed) == P.render(P.parse(P.render(parsed))), source  # render and parse are inverses
    assert F.same(P.parse(P.render(parsed)), parsed), source
assert P.render(P.parse("import numpy.ma\nnumpy.ma.masked")) == "import numpy.ma\nnumpy.ma.masked"
assert P.render(P.parse("x if a else y if b else z")) == "x if a else y if b else z"
assert P.render(P.parse("(x if a else y) if b else z")) == "(x if a else y) if b else z"
assert P.render(P.parse("import a, b as c\nfrom d import e, f as g\n1")) == (
    "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1")
assert [P.render(P.constant(v)) for v in [math.nan, math.inf, -math.inf, -2]] == [
    "float('nan')", "float('inf')", "-float('inf')", "-2"]
assert P.render(P.binop("**", -2, 2)) == "(-2) ** 2" and P.render(P.unaryop("-", P.binop("**", 2, 2))) == "-2 ** 2"
assert P.render(P.call("np.ma.getmaskarray", P.name("x"))) == "np.ma.getmaskarray(x)"  # a dotted function
built = P.importfrom("m", "f", P.let_("a", P.subscript(P.name("t"), "k"), P.ifexp(
    P.boolop("and", P.compare("<", P.name("a"), 1), P.unaryop("not", P.attribute(P.name("a"), "x"))),
    P.binop("+", P.call("f", P.name("a")), 2.5), P.constant(b""))), alias="g")
assert F.same(built, P.parse("from m import f as g\n(lambda a: f(a) + 2.5 if a < 1 and not a.x else b'')(t['k'])"))
assert P.render(P.attribute(P.binop("+", 1, 2), "real")) == "(1 + 2).real"
with raises(ValueError, match="an import can only enclose the whole expression"):
    P.render(P.binop("+", 1, P.import_("math", 2)))
for source, message in [
    ("f(x=1)", "cannot parse 'f(x=1)': keyword arguments are not supported"),
    ("a < b < c", "cannot parse 'a < b < c': a comparison has one operator"),
    ("(lambda a, b: a)(1, 2)", "cannot parse '(lambda a, b: a)(1, 2)': a let binds one name"),
    ("(lambda a=1: a)(2)", "a let binds one name"),
    ("x[0]", "cannot parse 'x[0]': a subscript's key must be a str"),
    ("None", "cannot parse 'None': only native constants are supported"),
    ("a @ b", "cannot parse 'a @ b': not supported in an expression"),
    ("a is b", "cannot parse 'a is b': not supported in an expression"),
    ("{1}", "cannot parse '{1}': not supported in an expression"),
    ("x = 1\nx", "cannot parse 'x = 1': only imports may precede the expression"),
    ("from . import a\na", "cannot parse 'from . import a': only imports may precede the expression"),
    ("from m import *\na", "cannot parse 'from m import *': import * binds names that cannot be known"),
]:
    with raises(ValueError, match=message):
        P.parse(source)
for source in ["", "import math"]:
    with raises(ValueError, match="the source must end with an expression"):
        P.parse(source)
assert P.import_("numpy.ma", 1).binds() == ("numpy",) and P.importfrom("m", "n", 1, alias="k").binds() == ("k",)
assert P.import_("not a module", 1).validate() == ["an import's module must be a dotted name, got 'not a module'"]
assert P.import_("m", 1, alias="a b").validate() == ["an import's alias must be an identifier, got 'a b'"]
assert P.importfrom("m.", "x y", 1).validate() == ["an importfrom's module must be a dotted name, got 'm.'",
                                                    "an importfrom's name must be an identifier, got 'x y'"]
assert P.importfrom("m", "n", 1, alias="").validate() == ["an importfrom needs an alias"]  # empty, not absent
assert P.let_("a b", 1, 2).validate() == ["a let's name must be an identifier, got 'a b'"]
assert P.attribute(P.name("x"), "1").validate(bound={"x"}) == ["an attribute must be an identifier, got '1'"]
assert P.importfrom("m", "n", P.name("n")).validate() == [] and P.DIALECT.kinds()["import"](7, None, P.constant(1)).validate() == [
    "an import's module must be a str, got int"]
assert P.binop("@", 1, 2).validate(core=True) == ["'@' is not a core operation"]
assert P.DIALECT.infer(P.parse("1 + 2.5")) is PD.Float and P.DIALECT.infer(P.parse("True + 1")) is PD.Int
assert P.DIALECT.infer(P.parse("'a' + 'b'")) is PD.Str and P.DIALECT.infer(P.parse("-2.0")) is PD.Float
assert P.DIALECT.infer(P.parse("2 ** 3")).name() == "int | float" and P.DIALECT.infer(P.parse("not 1")) is PD.Bool
assert P.DIALECT.infer(P.parse("1 < 2.5")) is PD.Bool and P.DIALECT.infer(P.parse("x.a"), {"x": D.Anything}) is D.Anything
with raises(TypeError, match="< cannot take (str, int)"):
    P.DIALECT.infer(P.parse("'a' < 1"))
assert PD.of(b"") is PD.Bytes and PD.Numeric.includes(PD.Bool)

## DIA-02 · Python evaluates by Python's rules, in a scope that allowlists what an expression may reach

In [ ]:
def ev(source, variables=None, **scope):
    return PV.OfAny(P.parse(source), PV.Scope(variables, **scope))


assert ev("0 or 'x'") == "x" and ev("'' and 1") == "" and ev("1 and 2") == 2  # an operand, not a bool
assert ev("1 == 1.0") is True and ev("True + 1") == 2 and ev("7 // 2 % 3 - 2 ** 2 / 4") == -1.0
assert ev("a if a > 1 else missing", {"a": 2}) == 2 and ev("(lambda a: a * a)(3)") == 9
assert ev("t.age", {"t": ann}) == 30 and ev("hasattr(t, 'email')", {"t": bob}) is False  # properties are attributes
assert ev("getattr(t, 'name')", {"t": ann}) == "Ann" and ev("t['x']", {"t": {"x": 1}}) == 1
assert ev("hasattr(t, '_secret')", {"t": ann}) is False
assert ev("abs(-2) + len('abc') + max(1, 2)") == 7 and ev("f(2)", {"f": lambda x: x + 1}) == 3
with raises(AttributeError, match="'Person' object has no attribute 'email'"):
    ev("t.email", {"t": bob})
with raises(AttributeError, match="attribute '__class__' is private"):
    ev("t.__class__", {"t": ann})
with raises(NameError, match="name 'x' is not defined"):
    ev("x")
with raises(NameError, match="name 'len' is not defined"):
    ev("len('a')", builtins={})


def shout(text):
    return text.upper()


with raises(TypeError, match="calling 'shout' is not allowed by the scope"):
    ev("t['f']('a')", {"t": {"f": shout}})  # only what the scope provides, not what values hold
with raises(TypeError, match="calling 'int' is not allowed by the scope"):
    ev("t()", {"t": 3})

numpy = {"numpy": np}
columns = {"age": np.array([30, 10, 70]), "email": np.ma.array(["a", "b", "c"], mask=[False, False, True])}
rule = ("import numpy as np\nfrom numpy.ma import getmaskarray\n"
        "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))")
assert ev(rule, {"t": columns}, modules=numpy).tolist() == [True, False, False]  # vectorized, with masks
assert ev("import numpy.ma\nnumpy.ma.masked", modules=numpy) is np.ma.masked  # binds numpy
assert ev("import numpy.ma as ma\nma.masked", modules=numpy) is np.ma.masked and ev("import numpy\nnumpy.pi",
                                                                                    modules=numpy) == math.pi
with raises(ImportError, match="import of 'numpy' is not allowed by the scope"):
    ev(rule, {"t": columns})  # nothing is imported unless the scope allows it
with raises(ImportError, match="import of 'os' is not allowed by the scope"):
    ev("import os\nos.getcwd()", modules=numpy)
with raises(ImportError, match="No module named 'numpy.nope'"):
    ev("import numpy.nope\n1", modules=numpy)
with raises(ImportError, match="No module named 'numpy._core'"):
    ev("import numpy._core\n1", modules=numpy)
with raises(ImportError, match="cannot import name 'nope' from 'numpy'"):
    ev("from numpy import nope\n1", modules=numpy)
with raises(ImportError, match="cannot import name '_core' from 'numpy'"):
    ev("from numpy import _core\n1", modules=numpy)
assert PV.OfAny(P.parse("x + 1"), {"x": 1}) == 2  # a mapping is the variables of a scope

# Python's rules, case by case. Here they run against Python itself; the TypeScript suite runs the same table against
# its model of them.
import math as math_module

n, NAN, INF = P.name, float("nan"), float("inf")
modules = {"math": math_module, "numpy": np}
VALUES = [  # (expression, variables, value): the value and its type
    (P.unaryop("not", n("v")), {"v": None}, True), (P.unaryop("not", 0.0), {}, True), (P.unaryop("not", ""), {}, True),
    (P.unaryop("not", b""), {}, True), (P.unaryop("not", n("v")), {"v": []}, True), (P.unaryop("not", n("v")), {"v": {}}, True),
    (P.unaryop("not", NAN), {}, False), (P.unaryop("not", n("v")), {"v": {"k": 1}}, False),
    (P.compare("<", 1, 1.5), {}, True), (P.compare(">", 2, 1.5), {}, True), (P.compare("==", 1, 1.5), {}, False),
    (P.compare("==", P.binop("+", P.binop("**", 2, 60), 1), P.call("float", P.binop("**", 2, 60))), {}, False),
    (P.compare("<", 1, INF), {}, True), (P.compare(">", 1, -INF), {}, True), (P.compare("<", 1, NAN), {}, False),
    (P.compare("<", 1.5, 2), {}, True), (P.compare("==", NAN, NAN), {}, False), (P.compare("==", 3, 3.0), {}, True),
    (P.compare("<", 3, 3.5), {}, True), (P.compare("<", 2.5, 1.5), {}, False), (P.compare("<", True, 2), {}, True),
    (P.compare("<", "a", "b"), {}, True), (P.compare("<", b"a", b"b"), {}, True), (P.compare("==", b"a", b"a"), {}, True),
    (P.compare("!=", "a", 1), {}, True),
    (P.binop("+", "ab", "c"), {}, "abc"), (P.binop("+", b"a", b"b"), {}, b"ab"), (P.binop("*", "ab", 2), {}, "abab"),
    (P.binop("*", 2, b"x"), {}, b"xx"), (P.binop("*", "ab", -1), {}, ""), (P.binop("//", -7, 2), {}, -4),
    (P.binop("%", -7, 2), {}, 1), (P.binop("%", 7, -2), {}, -1), (P.binop("%", -7.5, 2), {}, 0.5), (P.binop("%", 7.5, 2), {}, 1.5),
    (P.binop("//", 7.5, 2), {}, 3.0), (P.binop("**", 2, -1), {}, 0.5), (P.binop("**", 2.0, 3), {}, 8.0),
    (P.binop("+", 1.5, 1), {}, 2.5), (P.binop("*", 1.5, 2), {}, 3.0), (P.binop("-", 3, 1.5), {}, 1.5), (P.binop("/", 1, 2), {}, 0.5),
    (P.unaryop("-", True), {}, -1), (P.unaryop("+", True), {}, 1), (P.unaryop("-", 1.5), {}, -1.5), (P.unaryop("+", 1.5), {}, 1.5),
    (P.call("abs", True), {}, 1), (P.call("abs", -1.5), {}, 1.5), (P.call("bool", ""), {}, False), (P.call("float", 1), {}, 1.0),
    (P.call("float", " 1e3 "), {}, 1000.0), (P.call("float", "inf"), {}, INF), (P.call("float", "-inf"), {}, -INF),
    (P.call("str", P.call("float", "nan")), {}, "nan"), (P.call("int", " 42 "), {}, 42), (P.call("int", 2.7), {}, 2),
    (P.call("int", -2.7), {}, -2), (P.call("int", True), {}, 1), (P.call("len", b"ab"), {}, 2),
    (P.call("len", n("v")), {"v": [1, 2]}, 2), (P.call("len", n("v")), {"v": {"a": 1}}, 1), (P.call("len", "héllo"), {}, 5),
    (P.call("max", 1, 2.5), {}, 2.5), (P.call("min", 3, 1, 2), {}, 1), (P.call("round", 2.5), {}, 2),
    (P.call("round", 3.5), {}, 4), (P.call("round", -0.5), {}, 0), (P.call("round", 7), {}, 7), (P.call("str", 1.0), {}, "1.0"),
    (P.call("str", b"x"), {}, "b'x'"), (P.call("str", n("v")), {"v": None}, "None"), (P.call("str", True), {}, "True"),
    (P.call("str", 7), {}, "7"), (P.call("str", "s"), {}, "s"), (P.import_("math", P.call("math.floor", 2.5)), {}, 2),
    (P.unaryop("-", False), {}, 0), (P.unaryop("not", n("v")), {"v": ann}, False), (P.compare("==", 1.5, 1.5), {}, True),
    (P.binop("/", 1.5, 2), {}, 0.75),
]
for expression, variables, expected in VALUES:
    value = PV.OfAny(expression, PV.Scope(variables, modules=modules))
    assert value == expected and type(value) is type(expected), (P.render(expression), value, expected)
ERRORS = [  # (expression, variables, error, message)
    (P.compare("<", "a", 1), {}, TypeError, "'<' not supported between instances of 'str' and 'int'"),
    (P.binop("*", "a", 1.5), {}, TypeError, "can't multiply sequence by non-int of type 'float'"),
    (P.binop("-", "a", 1), {}, TypeError, "unsupported operand type(s) for -: 'str' and 'int'"),
    (P.binop("/", 1, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("//", 1.5, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("%", 1, 0), {}, ZeroDivisionError, "division by zero"),
    (P.binop("**", 0.0, -1), {}, ZeroDivisionError, "zero to a negative power"),
    (P.unaryop("-", "a"), {}, TypeError, "bad operand type for unary -: 'str'"),
    (P.call("abs", "a"), {}, TypeError, "bad operand type for abs(): 'str'"),
    (P.call("float", "x"), {}, ValueError, "could not convert string to float: 'x'"),
    (P.call("int", "x"), {}, ValueError, "invalid literal for int() with base 10: 'x'"),
    (P.call("int", NAN), {}, ValueError, "cannot convert float NaN to integer"),
    (P.call("int", INF), {}, OverflowError, "cannot convert float infinity to integer"),
    (P.call("len", 1), {}, TypeError, "object of type 'int' has no len()"),
    (P.call("max", 1, "a"), {}, TypeError, "'>' not supported between instances of 'str' and 'int'"),
    (P.call("round", "x"), {}, TypeError, "type str doesn't define __round__ method"),
    (P.call("float", n("v")), {"v": None}, TypeError, "float() argument must be a string or a real number, not 'NoneType'"),
    (P.call("int", n("v")), {"v": None}, TypeError,
     "int() argument must be a string, a bytes-like object or a real number, not 'NoneType'"),
    (P.import_("numpy", P.attribute(n("numpy"), "nope")), {}, AttributeError, "module 'numpy' has no attribute 'nope'"),
    (P.subscript(n("t"), "y"), {"t": {"x": 1}}, KeyError, "'y'"),
    (P.subscript(n("t"), "x"), {"t": 3}, TypeError, "'int' object is not subscriptable"),
    (P.call(P.subscript(n("t"), "f")), {"t": {"f": lambda: 1}}, TypeError, "calling '<lambda>' is not allowed by the scope"),
]
for expression, variables, error, message in ERRORS:
    with raises(error, match=message):
        PV.OfAny(expression, PV.Scope(variables, modules=modules))
assert PV.OfAny(P.importfrom("numpy.ma", "masked", n("masked")), PV.Scope(modules=modules)) is np.ma.masked

## DIA-03 · Matlab: operators, fields and `isfield`, rendered as MATLAB source

In [ ]:
s = M.identifier("s")
rule = M.binary("&&", M.binary(">=", M.field(s, "age"), 18), M.call("isfield", s, "email"))
assert M.render(rule) == 's.age >= 18 && isfield(s, "email")'
assert M.render(M.binary("-", 1, M.binary("-", 2, 3))) == "1 - (2 - 3)" and M.render(
    M.binary("-", M.binary("-", 1, 2), 3)) == "1 - 2 - 3"
assert M.render(M.binary(".*", M.binary("+", 1, 2), M.unary("-", M.unary("~", s)))) == "(1 + 2) .* -~s"
assert M.render(M.unary("-", M.binary("+", 1, 2))) == "-(1 + 2)" and M.render(M.unary("~", -1)) == "~-1"
assert M.render(M.binary("||", M.binary("&&", True, False), M.binary("&&", 1, 0))) == "true && false || 1 && 0"
assert [M.render(M.constant(v)) for v in [math.nan, math.inf, -math.inf, -2.5, 'say "hi"', 7]] == [
    "NaN", "Inf", "-Inf", "-2.5", '"say ""hi"""', "7"]
assert M.render(M.field(M.field(s, "a"), "b")) == "s.a.b"
assert M.DIALECT.infer(rule, {"s": MD.Struct}) is MD.Logical
assert M.DIALECT.infer(M.binary("+", "a", 1)) is MD.String and M.DIALECT.infer(M.binary("+", True, 1)) is MD.Double
assert MD.Double.contains(1) and MD.Struct.contains({}) and not MD.Struct.contains("s") and MD.of(1) is MD.Double
assert M.binary("^", 1, 2).validate(core=True) == ["'^' is not a core operation"]

## DIA-04 · Matlab evaluates two-valued, converts numbers and logicals, and has no unknown

In [ ]:
assert MV.OfAny(rule, {"s": ann}) is True and MV.OfAny(rule, {"s": bob}) is False
assert MV.OfAny(rule, {"s": {"age": 20, "email": "x"}}) is True  # a mapping is a struct
assert MV.OfAny(M.field(s, "age"), {"s": ann}) == 30.0 and type(MV.OfAny(18)) is float  # numbers are doubles
assert MV.OfAny(M.binary("==", 1, True)) is True and MV.OfAny(M.binary("+", True, 1)) == 2.0
assert MV.OfAny(M.binary("<", "apple", "banana")) is True and MV.OfAny(M.binary("==", "5", 5)) is True
assert MV.OfAny(M.binary("==", "true", True)) is True and MV.OfAny(M.binary("~=", "1.5", 1.5)) is False
assert MV.OfAny(M.binary("+", "a", 1)) == "a1" and MV.OfAny(M.binary("+", "x", 0.125)) == "x0.125"
assert [MV.OfAny(M.binary("+", "x", v)) for v in [math.nan, math.inf, -math.inf, 12345.6, 1.5e-7, 123456.7, True,
                                                   False]] == [
    "xNaN", "xInf", "x-Inf", "x12346", "x1.5e-07", "x1.2346e+05", "xtrue", "xfalse"]  # numbers as MATLAB writes them
assert [MV.OfAny(M.binary(op, 2, 3)) for op in ["~=", "<=", ">", ">=", "-", ".*"]] == [True, True, False, False,
                                                                                        -1.0, 6.0]
assert MV.OfAny(M.binary("&&", False, M.identifier("missing"))) is False  # short-circuits
assert MV.OfAny(M.binary("||", 2, M.identifier("missing"))) is True and MV.OfAny(M.binary("||", 0, 0)) is False
assert MV.OfAny(M.unary("~", 0)) is True and MV.OfAny(M.unary("-", True)) == -1.0
assert MV.OfAny(M.call("isfield", 3, "x")) is False and MV.OfAny(M.call("isfield", s, 1), {"s": ann}) is False
with raises(KeyError, match='Unrecognized field name "email".'):
    MV.OfAny(M.field(s, "email"), {"s": bob})
with raises(TypeError, match="Dot indexing is not supported for variables of this type."):
    MV.OfAny(M.field(s, "age"), {"s": 3})
with raises(NameError, match="Unrecognized function or variable 'missing'."):
    MV.OfAny(M.identifier("missing"))
with raises(TypeError, match="Operands to the && operator must be convertible to logical scalar values."):
    MV.OfAny(M.binary("&&", "yes", True))
with raises(ValueError, match="NaN's cannot be converted to logicals."):
    MV.OfAny(M.unary("~", math.nan))
with raises(TypeError, match="Operator '-' is not supported for operands of type 'string' and 'double'."):
    MV.OfAny(M.binary("-", "a", 1))
with raises(TypeError, match="Operator '.*' is not supported for operands of type 'struct' and 'logical'."):
    MV.OfAny(M.binary(".*", s, True), {"s": ann})
with raises(TypeError, match="Operator '-' is not supported for operands of type 'string'."):
    MV.OfAny(M.unary("-", "a"))

## DIA-05 · Matlab resolves functions from imports, the path and packages that the scope provides

In [ ]:
geo = {"dist": lambda a, b: (a * a + b * b) ** 0.5, "area": lambda r: 3 * r * r}
scope = MV.Scope({"r": 2}, functions={"twice": lambda x: 2 * x}, packages={"geo": geo, "geo.more": {"area": lambda r: 0}})
wild = M.import_("geo.*", M.binary("+", M.call("dist", 3, 4), M.call("area", M.identifier("r"))))
assert M.render(wild) == "import geo.*\ndist(3, 4) + area(r)" and wild.validate(bound={"r"}) == []
assert wild.validate(bound={"r"}, core=True) == ["body: left: 'dist' is not a core operation",
                                                 "body: right: 'area' is not a core operation"]
assert MV.OfAny(wild, scope) == 17.0  # an int result is a double
assert MV.OfAny(M.import_("geo.dist", M.call("dist", 6, 8)), scope) == 10.0
assert MV.OfAny(M.call("geo.more.area", 1), scope) == 0.0 and MV.OfAny(M.call("twice", 4), scope) == 8.0
assert MV.OfAny(M.import_("geo.*", M.import_("geo.more.area", M.call("area", 5))), scope) == 0.0  # innermost wins
with raises(NameError, match="Unrecognized function or variable 'dist'."):
    MV.OfAny(M.call("dist", 3, 4), scope)  # not imported
with raises(ImportError, match="Import argument 'os.system' cannot be found or cannot be imported."):
    MV.OfAny(M.import_("os.system", 1), scope)
with raises(ImportError, match="Import argument 'geo.nope' cannot be found or cannot be imported."):
    MV.OfAny(M.import_("geo.nope", 1), scope)
with raises(ImportError, match="Import argument 'nope.*' cannot be found or cannot be imported."):
    MV.OfAny(M.import_("nope.*", 1), scope)
with raises(ValueError, match="an import can only enclose the whole expression"):
    M.render(M.binary("+", 1, M.import_("geo.*", 2)))
assert M.import_("geo", 1).validate() == ["an import's name must be pkg.name or pkg.*, got 'geo'"]
assert M.import_("geo.*.x", 1).validate() == ["an import's name must be pkg.name or pkg.*, got 'geo.*.x'"]
assert M.DIALECT.kinds()["import"](7, M.constant(1)).validate() == ["an import's name must be a str, got int"]

## DIA-06 · Excel: formulas with `LET`, functions, operators and fields

In [ ]:
r = X.name("r")
rule = X.let_("age", X.field(r, "age"), X.function("AND", X.infix(">=", X.name("age"), 18),
                                                     X.function("NOT", X.function("ISERROR", X.field(r, "email")))))
assert X.render(rule) == "=LET(age, r.age, AND(age >= 18, NOT(ISERROR(r.email))))"
assert X.render(X.infix("*", X.infix("+", 1, 2), X.prefix("-", X.infix("-", 3, 4)))) == "=(1 + 2) * -(3 - 4)"
assert X.render(X.infix("=", X.infix("-", 1, X.infix("-", 2, 3)), -1)) == "=1 - (2 - 3) = -1"
assert X.render(X.prefix("-", X.field(r, "first name"))) == "=-r.[first name]"
assert [X.render(X.constant(v)) for v in [True, False, 'say "hi"', math.inf, 2.5, 3]] == [
    "=TRUE", "=FALSE", '="say ""hi"""', "=#NUM!", "=2.5", "=3"]
assert rule.validate(bound={"r"}, core=True) == [] and X.function("VLOOKUP", 1, 2).validate(core=True) == [
    "'VLOOKUP' is not a core operation"]
assert X.DIALECT.infer(rule, {"r": XD.Record}) is XD.Logical and X.DIALECT.infer(X.infix("+", "1", True)) is XD.Number
assert XD.of(1) is XD.Number and XD.Errors.contains(XD.Error("#N/A")) and repr(XD.Error("#N/A")) == "#N/A"
assert XD.Record.contains(ann) and not XD.Record.contains(1)

## DIA-07 · Excel coerces, orders values of every type, and computes with errors

In [ ]:
Error = XV.Error
assert XV.OfAny(rule, {"r": ann}) is True and XV.OfAny(rule, {"r": bob}) is False  # ISERROR finds the #FIELD!
assert XV.OfAny(X.field(r, "email"), {"r": bob}) == Error("#FIELD!") and XV.OfAny(X.name("nope")) == Error("#NAME?")
assert XV.OfAny(X.field(r, "age"), {"r": {"age": 5}}) == 5.0 and XV.OfAny(X.field(r, "x"), {"r": 3}) == Error("#VALUE!")
assert XV.OfAny(X.field(X.field(r, "a"), "b"), {"r": {}}) == Error("#FIELD!")  # errors propagate
assert XV.OfAny(X.infix("+", "1", True)) == 2.0 and XV.OfAny(X.infix("*", "x", 1)) == Error("#VALUE!")  # coercion
assert XV.OfAny(X.infix("-", 1, X.infix("+", X.name("u"), 1))) == Error("#NAME?")
assert XV.OfAny(X.prefix("-", "2")) == -2.0 and XV.OfAny(X.prefix("-", X.name("u"))) == Error("#NAME?")
assert XV.OfAny(X.infix(">", "a", 1)) is True and XV.OfAny(X.infix(">", True, "z")) is True  # number < text < logical
assert XV.OfAny(X.infix("=", "a", "A")) is True and XV.OfAny(X.infix("<>", 1, "1")) is True  # case; no coercion
assert [XV.OfAny(X.infix(op, 2, 3)) for op in ["<", "<=", ">=", "-"]] == [True, True, False, -1.0]
assert XV.OfAny(X.infix("=", 2, 2)) is True and XV.OfAny(X.infix("+", False, 1)) == 1.0
assert XV.OfAny(X.infix("+", "inf", 1)) == Error("#VALUE!") and XV.OfAny(X.infix("+", " 1e3 ", 1)) == 1001.0  # decimal text
assert XV.OfAny(X.infix("+", r, 1), {"r": {}}) == Error("#VALUE!")
assert XV.OfAny(X.infix("=", X.name("u"), 1)) == Error("#NAME?") and XV.OfAny(X.infix("=", r, 1), {"r": {}}) == Error(
    "#VALUE!")
assert XV.OfAny(X.function("AND", 1, True)) is True and XV.OfAny(X.function("OR", 0, False)) is False
assert XV.OfAny(X.function("AND", False, X.name("u"))) == Error("#NAME?")  # AND evaluates every argument
assert XV.OfAny(X.function("OR", "yes", True)) == Error("#VALUE!") and XV.OfAny(X.function("NOT", 0)) is True
assert XV.OfAny(X.function("NOT", X.name("u"))) == Error("#NAME?")
assert XV.OfAny(X.function("IF", True, 1, X.name("u"))) == 1.0 and XV.OfAny(X.function("IF", 0, 1, "no")) == "no"
assert XV.OfAny(X.function("IF", X.name("u"), 1, 2)) == Error("#NAME?")  # IF evaluates only the branch it takes
assert XV.OfAny(X.function("ISERROR", 1)) is False and XV.OfAny(X.let_("x", 2, X.infix("*", X.name("x"), 3))) == 6.0

## DIA-08 · Excel reads cells from workbooks, and add-in functions from the workbook's add-ins

In [ ]:
rates = XV.Workbook(sheets={"Rates": {"B2": 0.5}}, name="Rates.xlsx")
book = XV.Workbook({"base": 10}, {"Sheet1": {"a1": 2, "$A$2": 3}, "My Sheet": {"C3": "x"}}, books={"Rates.xlsx": rates},
                   add_ins={"double": lambda v: v * 2})
total = X.infix("+", X.infix("*", X.cell("A1"), X.cell("B2", "Rates", "Rates.xlsx")), X.function("DOUBLE", X.name("base")))
assert X.render(total) == "=A1 * [Rates.xlsx]Rates!B2 + DOUBLE(base)" and XV.OfAny(total, book) == 21.0
assert total.validate(bound={"base"}) == [] and total.validate(bound={"base"}, core=True) == [
    "right: 'DOUBLE' is not a core operation"]
assert X.render(X.cell("C3", "My Sheet")) == "='My Sheet'!C3" and XV.OfAny(X.cell("C3", "My Sheet"), book) == "x"
assert X.render(X.cell("A1", "Bob's", "My Book.xlsx")) == "='[My Book.xlsx]Bob''s'!A1"
assert XV.OfAny(X.cell("$a$2"), book) == 3.0 and XV.OfAny(X.cell("Z9"), book) == 0.0  # an empty cell is 0
assert XV.OfAny(X.cell("A1", "Sheet1", "Book1"), book) == 2.0  # this book, named
assert XV.OfAny(X.cell("A1", "Nope"), book) == Error("#REF!") and XV.OfAny(X.cell("A1", "S", "Other.xlsx"), book) == Error(
    "#REF!")
assert XV.OfAny(X.function("NOPE", 1), book) == Error("#NAME?") and XV.OfAny(X.function("double", X.name("u")), book) == Error(
    "#NAME?")  # add-ins ignore case; errors in their arguments propagate
assert XV.OfAny(X.cell("A1"), XV.Workbook()) == Error("#REF!")  # a workbook has no sheet until given one
assert XV.OfAny(X.name("x"), {"x": 1}) == 1.0 and XV.Workbook().sheet == "Sheet1"
with raises(ValueError, match="not a cell address: 'total'"):
    XV.Workbook(sheets={"Sheet1": {"total": 1}})
assert X.cell("A0").validate() == ["a cell's address must be a column and a row, like A1, got 'A0'"]
assert X.cell("A1", None, "B.xlsx").validate() == ["a cell in another book needs a sheet"]
assert X.DIALECT.infer(X.cell("A1")) is D.Anything and X.address("$b$12") == "B12" and X.address("B") is None